In [40]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

In [41]:
df = pd.read_csv('phishing_Email.csv')
df

,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email
...,...,...,...
18645,18646,date a lonely housewife always wanted to date ...,Phishing Email
18646,18647,request submitted : access request for anita ....,Safe Email
18647,18648,"re : important - prc mtg hi dorn & john , as y...",Safe Email
18648,18649,press clippings - letter on californian utilit...,Safe Email


In [42]:
df['Email Type'].value_counts()

Email Type
Safe Email        11322
Phishing Email     7328
Name: count, dtype: int64

In [43]:
df_new = df.drop('Unnamed: 0', axis=1)

In [44]:
df_new.head()

,Email Text,Email Type
0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,the other side of * galicismos * * galicismo *...,Safe Email
2,re : equistar deal tickets are you still avail...,Safe Email
3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [45]:
texts = df["Email Text"].astype(str)
df["labels"] = df["Email Type"].map({
    "Safe Email": 0,
    "Phishing Email": 1
})
df["labels"] = df["labels"].astype("int32")
labels = df["labels"].values

In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

In [47]:
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [48]:
max_len = 100

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding="post")

In [49]:
model = Sequential([
    Embedding(input_dim=10000, output_dim=64, input_length=max_len),
    SimpleRNN(64),
    Dropout(0.5),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

c:\Users\rajir\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [50]:
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

In [52]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test_pad, y_test)
)

Epoch 1/10
467/467 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.8151 - loss: 0.3031 - val_accuracy: 0.7708 - val_loss: 0.4460
Epoch 2/10
467/467 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.8168 - loss: 0.2925 - val_accuracy: 0.7973 - val_loss: 0.3946
Epoch 3/10
467/467 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.8170 - loss: 0.2786 - val_accuracy: 0.7668 - val_loss: 0.4411
Epoch 4/10
467/467 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.8217 - loss: 0.2865 - val_accuracy: 0.7686 - val_loss: 0.4542
Epoch 5/10
467/467 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.8276 - loss: 0.2699 - val_accuracy: 0.7989 - val_loss: 0.4125
Epoch 6/10
467/467 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.8386 - loss: 0.2603 - val_accuracy: 0.7885 - val_loss: 0.4433
Epoch 7/10
467/467 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.9244 - loss: 0.1991 - val_accuracy: 0.8906 - val_loss: 0.3425
Epoch 8/10
467/467 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - accuracy: 0.8883 - loss: 0.2313 - val_acc

In [53]:
loss, acc = model.evaluate(X_test_pad, y_test)
print("Accuracy:", acc)

117/117 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7389 - loss: 0.5976
Accuracy: 0.7241286635398865


In [54]:
def predict_email(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_len, padding="post")
    pred = model.predict(pad)[0][0]

    if pred > 0.5:
        print("Phishing Email")
    else:
        print("Safe Email")

predict_email("Congratulations! Click here to claim reward")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 316ms/step
Safe Email


In [58]:
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential

In [55]:
y_train = np.array(y_train)
y_test = np.array(y_test)

In [59]:
model = Sequential([
    Embedding(input_dim=10000, output_dim=128, input_length=max_len),

    LSTM(128),

    Dropout(0.5),

    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

In [60]:
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

In [61]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test_pad, y_test)
)

Epoch 1/5
467/467 ━━━━━━━━━━━━━━━━━━━━ 24s 46ms/step - accuracy: 0.7424 - loss: 0.5027 - val_accuracy: 0.9134 - val_loss: 0.3321
Epoch 2/5
467/467 ━━━━━━━━━━━━━━━━━━━━ 21s 45ms/step - accuracy: 0.9006 - loss: 0.2840 - val_accuracy: 0.9547 - val_loss: 0.1574
Epoch 3/5
467/467 ━━━━━━━━━━━━━━━━━━━━ 21s 46ms/step - accuracy: 0.9448 - loss: 0.1661 - val_accuracy: 0.9584 - val_loss: 0.1391
Epoch 4/5
467/467 ━━━━━━━━━━━━━━━━━━━━ 26s 56ms/step - accuracy: 0.9734 - loss: 0.0939 - val_accuracy: 0.9568 - val_loss: 0.1334
Epoch 5/5
467/467 ━━━━━━━━━━━━━━━━━━━━ 22s 47ms/step - accuracy: 0.9179 - loss: 0.2078 - val_accuracy: 0.9324 - val_loss: 0.1944


In [62]:
loss, acc = model.evaluate(X_test_pad, y_test)
print("Accuracy:", acc)


117/117 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9309 - loss: 0.1956
Accuracy: 0.9324396848678589


In [63]:
def predict_email(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_len, padding="post")

    pred = model.predict(pad)[0][0]

    if pred > 0.5:
        print("Phishing Email")
    else:
        print("Safe Email")


predict_email("Congratulations! Click here to claim reward")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step
Phishing Email
